# 04 · Live viewer (A-line, envelope, M-mode)

`minus.live.LiveView` is a live object for Jupyter: run controls (start/stop, mode, input, gain) above an image that
updates in place while a background thread fires shots. Each frame shows the A-line (RF), its envelope in dB and an
M-mode image (the last 120 envelopes, depth vertical). Modes: `pe` (pulse + RF, ~23 frames/s with the SPI link and base64
output), `acq` (no pulse), `env` (envelope computed on the PIC), `envbp` (band-pass envelope on the PIC, ~36 frames/s), `tof` (`pe` + the PIC's time of flight as a
red marker). The same figure runs as a desktop window: `python3 -m minus.live` (see `--help`).

**While a view is running it owns the serial port** — call `view.stop()` before using `dev` in other cells.

Executed on board 3 with the bench transducer on input 25 (first echo ~43 µs = 33 mm), gain ×3.2. Modes: `pe`, `acq`,
`env`, `envbp` (band-pass envelope computed on the PIC), `tof` (`pe` + the time-of-flight marker).

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "library"))
from minus import Minus
from minus.live import LiveView
dev = Minus()
dev.gain(0, 1, 0)                   # x3.2: the first echo uses most of the range

Gain(leg=0, rf_ohm=2200.0, oa1=1.0, oa1_unity=True, oa2=3.2, oa2_unity=False, total=3.2, total_db=10.1)

In [2]:
view = LiveView(dev, mode="pe", in_=25)   # displays the controls + image

/home/kelu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Synchronous frames (what a headless run, e.g. `nbconvert --execute`, can show):

In [3]:
f = view.run(frames=15)
print(view.frames, "frames;", f.info)

15 frames; pe in 25: 4095 samples, 40.2 ns, spi@12.50MHz 5718us


Background thread: the image keeps updating while the cell has returned (press **run** in the widget, or):

In [4]:
view.start(); time.sleep(3); view.stop()
print(view.frames, "frames so far;", view.w_status.value)

31 frames so far; stopped after 31 frames


Envelope mode (the PIC sends 511 envelope values instead of 4095 samples):

In [5]:
for mode in ("env", "envbp"):
    view.src.mode = mode; view.src.dec = 8
    view.run(frames=10); print(view.w_status.value)

frame 41: env dec 8 x1 shots, 511 pts, spi@12.50MHz


frame 51: envbp dec 8 x1 shots, 511 pts, spi@12.50MHz


Time-of-flight mode (red marker = the PIC's first threshold crossing after the blank):

In [6]:
view.src.mode = "tof"; view.src.thr = 60; view.src.blank_ns = 25000
view.run(frames=10); print(view.w_status.value)

frame 61: tof in 25: 4095 samples, 40.2 ns, spi@12.50MHz 5719us | tof 32.6 mm (thr 60)


In [7]:
view.stop()
dev.close()